# Egyptian Used-Car Valuation Engine

**Covers:** Future Value Prediction, Value Retention Score (VRS), Investment Rating, New-Model Potential, EN/AR Explainer.

## 1. Importing libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import category_encoders as ce
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.dpi'] = 110

CSV_PATH = "./data/hatla2ee_scraped_data_cleaned.csv"
CURRENT_YEAR = 2026

In [2]:
df = pd.read_csv(CSV_PATH)
df.head()

,Price,Make,Model,Year,Mileage (km),Color,Transmission,Fuel Type,Body Type,Condition,City,Governorate,Age,mileage_band
0,1950000,Audi,Q3,2021,56000.0,White,Automatic,Gas,SUV,Used,Tagamo3 - New Cairo,Cairo,5,20-60K
1,150000,Daewoo,Matiz,1999,180.0,Other,Manual,Gas,Hatchback,Used,Damietta,Damietta,27,0-20K
2,2700000,Mercedes,C 180,2023,55000.0,Gray,Automatic,Gas,Sedan,Used,Tagamo3 - New Cairo,Cairo,3,20-60K
3,2150000,Mini,Cooper,2023,6000.0,Red,Automatic,Gas,Convertible,Used,Tagamo3 - New Cairo,Cairo,3,0-20K
4,1300000,Jetour,X90,2024,40000.0,Black,Automatic,Gas,MPV,Used,Nasr city,Cairo,2,20-60K


In [3]:
def annual_mileage_estimate(d):
    sub = d[d["Age"] > 0]
    return float((sub["Mileage (km)"] / sub["Age"]).median())

KM_PER_YEAR = annual_mileage_estimate(df)
print(f"Estimated typical usage: {KM_PER_YEAR:,.0f} km/year")

Estimated typical usage: 13,000 km/year


## 2. Methodology

| Component | Method | Caveat |
|---|---|---|
| **Price prediction** | Random Forest on Make/Model/Age/Mileage/Fuel/Body/Governorate/Transmission. Categorical encoding: OHE for low-cardinality fields; BinaryEncoder for Make/ModelClean (high-cardinality). Rare models (<10 listings) collapse into an `Other` bucket per brand. Target: `log(price)`. | "Future value" = what a car *this age* sells for *today*. We assume today's age-price curve holds — standard cross-sectional practice, but still an assumption. |
| **Confidence band** | 10th–90th percentile spread across individual trees in the forest. | Reflects listing price *dispersion*, not validated forecast accuracy. The `evaluate_model()` cell quantifies this via MAE, MAPE, R², and interval coverage on a held-out 10% test split. |
| **VRS (Value Retention Score)** | Model-level, geometric: `(p_old/p_young)^(1/age_diff) × 100`, scaled 0–100 with anchors lo=85%, hi=97% annual retention. Needs ≥50 listings and ≥3 in each of the 0–2y and 9–12y age bands. Falls back to market median if a specific model can't be scored. | Mixes true depreciation resistance with nominal EGP inflation (no CPI adjustment applied). A high VRS partly reflects recent EGP devaluation inflating younger-car prices. |
| **Investment rating** | Rule-based: VRS threshold + listing price vs. predicted fair value. | Not independently modeled. |

## 3. Price Prediction Model

In [4]:
def fit_price_model(d):
    feats = ["Make", "Model", "Age", "Mileage (km)", "Fuel Type",
             "Body Type", "Governorate", "Transmission"]
    dd = d.dropna(subset=feats).copy()

    model_counts = dd["Model"].value_counts()
    dd["ModelClean"] = np.where(dd["Model"].map(model_counts) >= 10, dd["Model"], "Other")

    cat_cols1 = ["Fuel Type", "Body Type", "Governorate", "Transmission"]
    cat_cols2 = ["Make", "ModelClean"]
    num_cols = ["Age", "Mileage (km)"]

    encoder1 = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder2 = ce.BinaryEncoder(cols=cat_cols2, handle_unknown="value")

    Xcat1 = encoder1.fit_transform(dd[cat_cols1])
    Xcat2 = encoder2.fit_transform(dd[cat_cols2])
    
    X = np.hstack([dd[num_cols].values, Xcat1, Xcat2])
    y = np.log1p(dd["Price"].values)

    rf = RandomForestRegressor(n_estimators=700, max_depth=18, min_samples_leaf=2, min_samples_split=2, max_features='log2', n_jobs=-1, random_state=42)
    rf.fit(X, y)
    print(f"Price model fit on {len(dd):,} rows")
    return rf, encoder1,encoder2, cat_cols1,cat_cols2, model_counts

train, test = train_test_split(df, test_size=0.1, random_state=42)
rf, encoder1, encoder2, cat_cols1, cat_cols2, model_counts = fit_price_model(train)

Price model fit on 10,058 rows


In [5]:
def predict_price(make, model, age, mileage, fuel="Gas", body="Sedan",
                   gov="Cairo", trans="Automatic"):
    model_clean = model if model_counts.get(model, 0) >= 10 else "Other"
    row = pd.DataFrame([{
        "Make": make, "ModelClean": model_clean, "Fuel Type": fuel,
        "Body Type": body, "Governorate": gov, "Transmission": trans
    }])
    
    Xcat1 = encoder1.transform(row[cat_cols1])
    Xcat2 = encoder2.transform(row[cat_cols2])
    Xcat = np.hstack([Xcat1, Xcat2])

    Xnum = np.array([[age, mileage]])
    X = np.hstack([Xnum, Xcat])

    # individual-tree spread = our confidence band
    tree_preds = np.array([t.predict(X)[0] for t in rf.estimators_])
    log_p10, log_p50, log_p90 = np.percentile(tree_preds, [10, 50, 90])
    return {"predicted": np.expm1(log_p50), "low": np.expm1(log_p10), "high": np.expm1(log_p90)}

# sanity check
predict_price("Kia", "Cerato", age=4, mileage=70_000)

{'predicted': np.float64(979915.5402314018),
 'low': np.float64(739021.6007521217),
 'high': np.float64(1398418.9194867392)}

In [6]:
def evaluate_model(test):
    actual_prices = test["Price"].values
    predicted_prices = []
    inside_interval = 0

    for _, row in test.iterrows():
        pred_dict = predict_price(
            make=row["Make"],
            model=row["Model"],
            age=row["Age"],
            mileage=row["Mileage (km)"],
            fuel=row.get("Fuel Type", "Gas"),
            body=row.get("Body Type", "Sedan"),
            gov=row.get("Governorate", "Cairo"),
            trans=row.get("Transmission", "Automatic"),
        )

        predicted_prices.append(pred_dict["predicted"])

        if pred_dict["low"] <= row["Price"] <= pred_dict["high"]:
            inside_interval += 1

    predicted_prices = np.array(predicted_prices)

    mae = mean_absolute_error(actual_prices, predicted_prices)
    rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
    r2 = r2_score(actual_prices, predicted_prices)
    mape = np.mean(np.abs((actual_prices - predicted_prices) / actual_prices)) * 100
    coverage_rate = (inside_interval / len(test)) * 100

    print("--- Model Performance Metrics ---")
    print(f"Mean Absolute Error (MAE): {mae:,.2f} EGP")
    print(f"Root Mean Squared Error (RMSE): {rmse:,.2f} EGP")
    print(f"MAPE (Average % Error): {mape:.2f}%")
    print(f"R² Score (Variance Explained): {r2:.4f}")
    print(f"Interval Coverage Rate: {coverage_rate:.2f}% (Target: ~85%)")

evaluate_model(test)

--- Model Performance Metrics ---
Mean Absolute Error (MAE): 315,974.63 EGP
Root Mean Squared Error (RMSE): 864,434.28 EGP
MAPE (Average % Error): 20.86%
R² Score (Variance Explained): 0.6656
Interval Coverage Rate: 84.97% (Target: ~85%)


## 4. Value Retention Score (VRS)

In [7]:
def build_model_level_vrs(d):
    dd = d.copy()
    dd["age_band"] = pd.cut(dd["Age"], bins=[-1, 2, 12], labels=["young", "old"])

    # Group by BOTH Make and Model
    counts = dd.groupby(["Make", "Model"]).size()
    solid_models = counts[counts >= 50].index

    rows = []
    for make, model in solid_models:
        sub = dd[(dd["Make"] == make) & (dd["Model"] == model)]

        young_cars = sub[sub["age_band"] == "young"]
        old_cars = sub[sub["age_band"] == "old"]

        if len(young_cars) < 3 or len(old_cars) < 3:
            continue

        p_young = young_cars["Price"].median()
        p_old = old_cars["Price"].median()

        age_young = young_cars["Age"].median()
        age_old = old_cars["Age"].median()

        age_diff = max(age_old - age_young, 1)

        annual_retention = (p_old / p_young) ** (1 / age_diff)

        rows.append(
            {
                "Make": make,
                "Model": model,
                "annual_retention_pct": annual_retention * 100,
                "median_age_diff": age_diff,
                "sample_size": len(sub),
            }
        )

    curves = pd.DataFrame(rows)

    lo, hi = 85, 97
    curves["VRS"] = ((curves["annual_retention_pct"] - lo) / (hi - lo) * 100).clip(
        0, 100
    )

    return curves.sort_values("VRS", ascending=False).reset_index(drop=True)


model_curves = build_model_level_vrs(df)
print(model_curves.head(20).to_string())

         Make       Model  annual_retention_pct  median_age_diff  sample_size        VRS
0     Hyundai  Elantra HD             96.757433              4.0           54  97.978606
1     Hyundai   Accent RB             96.181850              8.0           69  93.182080
2       Chery     Tiggo 3             95.650620              5.0           78  88.755171
3      Nissan      Sentra             95.367131              8.0           89  86.392761
4       Chery     Tiggo 7             95.265493              3.0           58  85.545776
5   Chevrolet       Optra             94.909959              7.0          195  82.582994
6      Nissan       Sunny             94.722408              5.0          255  81.020070
7    Mercedes       C 180             94.053630              5.0          194  75.446919
8          MG           5             93.874741              4.0          113  73.956175
9         Kia      Cerato             93.755801              8.0          144  72.965008
10    Hyundai  Tucson

In [8]:
def vrs_for(make, model=None):
    if model and "model_curves" in globals():
        matched_model = model_curves[
            (model_curves["Make"] == make) & (model_curves["Model"] == model)
        ]
        if not matched_model.empty:
            vrs = float(matched_model["VRS"].values[0])
            return vrs, "model-level"

    if "model_curves" in globals():
        market_median = float(model_curves["VRS"].median())
        return market_median, "market-median fallback"

    return None, "insufficient data"


def grade(vrs):
    if vrs is None:
        return "Unrated"
    if vrs >= 90:
        return "A+ Elite Retention (Loses ~5% per year)"
    if vrs >= 80:
        return "A Strong Retention (Loses ~6% per year)"
    if vrs >= 70:
        return "B Good Retention (Loses ~7% per year)"
    if vrs >= 50:
        return "C Average Retention (Loses ~8-9% per year)"
    return "D High Depreciation Risk (Loses >10% per year)"


vrs_score, status = vrs_for("Toyota", "Corolla")
print(
    f"Toyota Corolla -> VRS Score: {vrs_score:.2f} ({status}) -> Grade: {grade(vrs_score)}"
)

Toyota Corolla -> VRS Score: 71.21 (model-level) -> Grade: B Good Retention (Loses ~7% per year)


## 5. Future Value Projection

In [9]:
def future_value(make, model, current_age, mileage_now, horizons=(3, 5, 10), **kwargs):
    out = {}
    for h in horizons:
        future_age = current_age + h
        future_mileage = mileage_now + h * KM_PER_YEAR
        out[h] = predict_price(make, model, future_age, future_mileage, **kwargs)
    return out

print("2012 Kia Cerato bought at 600,000 EGP, today 300,000 km")
fv = future_value("Kia", "Cerato", current_age=CURRENT_YEAR - 2012, mileage_now=300_000)
for h, pred in fv.items():
    print(f"  +{h}y -> {pred['low']:,.0f} - {pred['high']:,.0f} EGP (median {pred['predicted']:,.0f})")

2012 Kia Cerato bought at 600,000 EGP, today 300,000 km
  +3y -> 422,720 - 677,323 EGP (median 589,587)
  +5y -> 353,390 - 658,385 EGP (median 544,719)
  +10y -> 243,205 - 634,083 EGP (median 438,141)


## 6. Investment Rating

In [10]:
def investment_rating(make, model, listing_price, age, mileage, **kwargs):

    pred = predict_price(make, model, age, mileage, **kwargs)
    vrs, basis = vrs_for(make, model)
    price_gap = (listing_price - pred["predicted"]) / pred["predicted"]

    if vrs is None:
        rating = "Insufficient data to rate"
    elif vrs >= 75 and price_gap <= -0.05:
        rating = "Buy"
    elif vrs < 45 or price_gap >= 0.15:
        rating = "Avoid"
    elif vrs >= 60 and abs(price_gap) < 0.10:
        rating = "Buy"
    else:
        rating = "Hold"

    return {
        "rating": rating,
        "grade": grade(vrs),
        "VRS": round(vrs, 1) if vrs is not None else None,
        "VRS_basis": basis,
        "fair_value_estimate": round(pred["predicted"]),
        "listing_vs_fair_value_%": round(price_gap * 100, 1),
    }


investment_rating("Kia", "Cerato", listing_price=600_000, age=CURRENT_YEAR-2012, mileage=300_000)

{'rating': 'Buy',
 'grade': 'B Good Retention (Loses ~7% per year)',
 'VRS': 73.0,
 'VRS_basis': 'model-level',
 'fair_value_estimate': 604868,
 'listing_vs_fair_value_%': np.float64(-0.8)}

## 7. EN/AR Explainer

In [11]:
def explain(make, model, lang="en"):
    vrs, basis = vrs_for(make, model)
    g = grade(vrs)
    vrs_txt = f"{vrs:.0f}/100" if vrs is not None else "not available"

    if vrs is not None:
        annual_ret = 85 + (vrs / 100) * (97 - 85)  # reverse the VRS scaling
        annual_loss = round(100 - annual_ret, 1)
        loss_txt = f"loses roughly {annual_loss}% of its value per year"
        loss_ar = f"تخسر نحو {annual_loss}% من قيمتها سنوياً"
    else:
        loss_txt = "insufficient data to estimate annual depreciation"
        loss_ar = "لا تتوفر بيانات كافية لتقدير الاستهلاك السنوي"

    commentary = {
        "A+ Elite Retention": "It consistently commands high demand on Hatla2ee and spare parts are widely available at competitive Egyptian prices.",
        "A Strong Retention": "It enjoys strong local demand and is among the easier models to resell quickly on Hatla2ee.",
        "B Good Retention": "It holds moderate demand in the Egyptian market with a reasonable resale timeline.",
        "C Average Retention": "Resale can take longer in the Egyptian market; buyers are more price-sensitive for this model.",
        "D High Depreciation Risk": "Weak local demand and/or high maintenance costs make this a slow reseller on Hatla2ee.",
        "Unrated (insufficient data)": "Too few listings across age bands to compute a reliable retention score for this model.",
    }
    note = commentary.get(g, "")

    model_str = model if model else ""

    if lang == "ar":
        commentary_ar = {
            "A+ Elite Retention": "تحظى بطلب مرتفع باستمرار على هتلاقي وقطع غيارها متاحة بأسعار تنافسية في السوق المصري.",
            "A Strong Retention": "تتمتع بطلب محلي قوي وتُعدّ من أسهل الموديلات إعادةً للبيع على هتلاقي.",
            "B Good Retention": "تحافظ على طلب معقول في السوق المصري مع جدول زمني مقبول لإعادة البيع.",
            "C Average Retention": "قد تستغرق إعادة بيعها وقتاً أطول في السوق المصري والمشترون أكثر حساسية للسعر.",
            "D High Depreciation Risk": "ضعف الطلب المحلي و/أو ارتفاع تكاليف الصيانة يجعلانها بطيئة البيع على هتلاقي.",
            "Unrated (insufficient data)": "عدد الإعلانات عبر الفئات العمرية غير كافٍ لحساب درجة احتفاظ موثوقة لهذا الموديل.",
        }
        note_ar = commentary_ar.get(g, "")
        return (
            f'{make} {model_str} حصلت على تقييم "{g}" '
            f"بمعدل VRS {vrs_txt} ({basis}). "
            f"هذا يعني أن السيارة {loss_ar}. "
            f"{note_ar}"
        )

    return (
        f'{make} {model_str} received a "{g}" rating '
        f"with a VRS of {vrs_txt} ({basis}). "
        f"This means the car {loss_txt}. "
        f"{note}"
    )


print(explain("Toyota", "Corolla", lang="en"))
print(explain("Toyota", "Corolla", lang="ar"))

Toyota Corolla received a "B Good Retention (Loses ~7% per year)" rating with a VRS of 71/100 (model-level). This means the car loses roughly 6.5% of its value per year. 
Toyota Corolla حصلت على تقييم "B Good Retention (Loses ~7% per year)" بمعدل VRS 71/100 (model-level). هذا يعني أن السيارة تخسر نحو 6.5% من قيمتها سنوياً. 


## 8. Export data

In [14]:
import joblib

joblib.dump(
    {
        "rf": rf,
        "encoder1": encoder1,
        "encoder2": encoder2,
        "cat_cols1": cat_cols1,
        "cat_cols2": cat_cols2,
        "model_counts": model_counts,
        "model_curves": model_curves,
        "KM_PER_YEAR": KM_PER_YEAR,
        # dropdown data
        "makes": sorted(train["Make"].dropna().unique().tolist()),
        "models": (train.groupby("Make")["Model"].unique().apply(list).to_dict()),
        "fuels": sorted(train["Fuel Type"].dropna().unique().tolist()),
        "bodies": sorted(train["Body Type"].dropna().unique().tolist()),
        "governorates": sorted(train["Governorate"].dropna().unique().tolist()),
        "transmissions": sorted(train["Transmission"].dropna().unique().tolist()),
    },
    "./exported/car_model.pkl",
)

print("Model exported successfully")

Model exported successfully
